# Demo Notebook

This notebook demonstrates how to run the Spatial Information Reconstruction Framework (SIRF) using the default configuration.

It loads a dataset from `data/<dataset_name>/bus.csv` and `data/<dataset_name>/branch.csv`, runs the reconstruction, and inspects the outputs in `results/<experiment_name>/`.

In [ ]:
# 1. Imports
import yaml
import pandas as pd
import os

import sirf

In [ ]:
# 2. Load config
with open('configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

config

In [ ]:
# 3. Run framework
sirf.run(config)   # will read bus.csv / branch.csv and save outputs to results/demo/

In [ ]:
# 4. Inspect results
iter_num = config["inference"]["iter_num"]
suffix = f"{int(config['inference']['retention_rate']*100)}p_{int(iter_num):04d}"

bus_path = "results/" + config['data']['dataset'] + "/bus.csv"
branch_path = "results/" + config['data']['dataset'] + "/branch.csv"

if os.path.exists(bus_path) and os.path.exists(branch_path):
    print("Loading final outputs (bus.csv, branch.csv)...")
    bus = pd.read_csv(bus_path)
    branch = pd.read_csv(branch_path)
else:
    print("Final outputs not found, loading aligned data...")
    bus = pd.read_csv(f"results/{config['data']['dataset']}/Aligned_data/Bus_aligned/Bus_aligned_{suffix}.csv")
    branch = pd.read_csv(f"results/{config['data']['dataset']}/Aligned_data/Branch_aligned/Branch_aligned_{suffix}.csv")

bus.head(), branch.head()

In [ ]:
# 5. Simple visualization of reconstructed nodes
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# 5-1. Load inference loss
inference_loss_path = os.path.join(
    "results",
    config['data']['dataset'],
    "Inferred_data",
    "Inference_loss",
    f"Inference_loss_{suffix}.csv"
)

inference_loss = pd.read_csv(inference_loss_path, header=None)
loss_values = inference_loss.iloc[0].tolist()

# 5-2. Prepare bus/branch data
if "Latitude" not in bus.columns and "Longitude" not in bus.columns:
    bus = bus.rename(
        columns={
            "Longitude_truth": "Longitude",
            "Latitude_truth": "Latitude"
        }
    )

# 5-3. Create subplots
fig = plt.figure(figsize=(14, 6))

# ---------------------------
# Left: Inference Loss
# ---------------------------
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(loss_values, label='Inference Loss', color='blue')
ax1.set_title(f"Inference Loss ({config['data']['dataset']})")
ax1.set_xlabel('Reclassification Steps')
ax1.set_ylabel('Average Target Loss')
ax1.legend()
ax1.grid(True)
ax1.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))

# ---------------------------
# Right: Reconstructed Grid with world map
# ---------------------------
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
ax2.set_title(f"Reconstructed Grid ({config['data']['dataset']})")

# Add map background
ax2.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.4)
ax2.add_feature(cfeature.BORDERS, linewidth=0.5)
ax2.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax2.add_feature(cfeature.LAKES, alpha=0.3)
ax2.add_feature(cfeature.RIVERS, alpha=0.3)

# Node display
ax2.scatter(bus['Longitude'], bus['Latitude'], s=30, c='blue',
            label='Nodes', transform=ccrs.PlateCarree())

# Branch connections
for _, row in branch.iterrows():
    n1 = bus[bus['Node'] == row['Node1']].iloc[0]
    n2 = bus[bus['Node'] == row['Node2']].iloc[0]
    ax2.plot([n1['Longitude'], n2['Longitude']],
             [n1['Latitude'], n2['Latitude']],
             'k-', alpha=0.5, transform=ccrs.PlateCarree())

ax2.legend(loc='lower left')

# Automatically adjust axes to fit data range
lon_min, lon_max = bus['Longitude'].min(), bus['Longitude'].max()
lat_min, lat_max = bus['Latitude'].min(), bus['Latitude'].max()
ax2.set_extent([lon_min - 1, lon_max + 1, lat_min - 1, lat_max + 1], crs=ccrs.PlateCarree())

plt.tight_layout()
plt.show()